# 07_rotated_mnist_multiseed — Validazione a 5 seed (source + adattamento), MNIST -> Rotated-MNIST 30°

Ripete l'adattamento a 2 bracci (`shot_im`, `u_sfan` -- **non** `epistemic_only`, rimosso dal
progetto: verificato che `code_v2/src/digits_adapt.py` non lo espone più prima di procedere)
già fatto in `06_rotated_mnist_adapt.ipynb` (che resta invariato, a singolo seed: training
source seed=0, adattamento seed=1), ma su **5 seed indipendenti** (`SEEDS = [0, 1, 2, 3, 4]`),
stessa convenzione già usata per i due multiseed precedenti (Amazon Reviews, digits). Stesso
angolo di rotazione del Notebook 6 (**30°**, non riselezionato per seed), stesso held-out
MNIST (`mnist_te[2000:4000]`). Qui il seed varia **sia il training del source model (MNIST
pulito) SIA l'adattamento**: per ciascun seed si riaddestra il modello MNIST da zero (stesso
schema del Notebook 3: `train_map`, 3 epoche fisse, nessun early stopping -- molto più
economico del training SVHN della pipeline digits), si rifitta la Laplace, si ricontrolla la
convergenza MC, si rigenera il target ruotato dalle feature del NUOVO modello (stesse
immagini, stessa rotazione di 30°, ma feature ricalcolate dal modello di quel seed).

**Nessuna duplicazione di codice**: `code_v2.src.bayesian_model.{SmallCNN, train_map, extract,
head_weights, augment, LastLayerLaplace}` e `code_v2.src.digits_adapt.adapt_target`
(invariati). Il fit di Laplace + controllo di convergenza, come per la pipeline digits, non
esiste come funzione condivisa per questo source MNIST: resta definito localmente in questo
notebook.

## 1. Stima del tempo totale, prima di lanciare i 5 seed per intero

Basata sui tempi misurati nel Notebook 06 a singolo seed, moltiplicati per 5: il benchmark
preliminare lì misura **0.584 s/step** (un singolo step full-batch isolato, N=2000,
`lr=1e-3` -- deviazione già documentata per collasso a `lr=1e-2`), con `ADAPT_STEPS=100`.
Training MNIST (Notebook 03: `train_map`, 3 epoche su un sottoinsieme di 20.000 immagini) e
fit di Laplace non sono cronometrati nel Notebook 06 (fisso `M=200`, nessuno sweep di
convergenza) -- restano stime approssimative qui sotto.

**Attenzione a come va letta questa stima**: un singolo step misurato subito dopo l'avvio
del kernel include il costo di warm-up (allocatori, thread BLAS, contesto CUDA), quindi
sovrastima il costo a regime. La Sezione 3 confronta stima e tempo osservato e quantifica
lo scarto.

In [1]:
EST_TRAIN_S = 15         # stima approssimativa (training MNIST, 3 epoche, non cronometrato nel Notebook 06)
EST_LAPLACE_S = 20       # stima approssimativa (fit Laplace + sweep di convergenza, assenti nel Notebook 06)
EST_STEP_S = 0.584       # da 06_rotated_mnist_adapt.ipynb, cella di benchmark (singolo step isolato)
N_ARMS, ADAPT_STEPS = 2, 100
N_SEEDS = 5

est_adapt_s = N_ARMS * ADAPT_STEPS * EST_STEP_S
est_per_seed_s = EST_TRAIN_S + EST_LAPLACE_S + est_adapt_s
est_total_s = est_per_seed_s * N_SEEDS

print(f"stima per seed: training~{EST_TRAIN_S}s + laplace/convergenza~{EST_LAPLACE_S}s + "
      f"adattamento({N_ARMS} bracci x {ADAPT_STEPS} step)={est_adapt_s:.0f}s  "
      f"= {est_per_seed_s:.0f}s (~{est_per_seed_s/60:.1f} min)")
print(f"stima TOTALE per {N_SEEDS} seed: {est_total_s:.0f}s (~{est_total_s/60:.1f} minuti)")
print()
print("(stima PRIMA di lanciare la cella lunga sotto, basata su un singolo step isolato:")
print(" il tempo osservato viene confrontato con questa stima subito dopo l'esecuzione)")

stima per seed: training~15s + laplace/convergenza~20s + adattamento(2 bracci x 100 step)=117s  = 152s (~2.5 min)
stima TOTALE per 5 seed: 759s (~12.7 minuti)

(stima PRIMA di lanciare la cella lunga sotto, basata su un singolo step isolato:
 il tempo osservato viene confrontato con questa stima subito dopo l'esecuzione)


## 2. Verifica di M_FIXED su più seed, prima di assumerlo fisso

Stessa cautela già usata per Amazon Reviews e per i digit: ricalcolo la convergenza per
ciascuno dei 5 seed invece di assumere quella del seed 0. **Anticipazione del risultato
(Sezione 3): `M_FIXED` NON è stabile fra seed** (250, 250, 100, 1000, 250 -- un outlier a
1000 su un solo seed) -- ricalcolarlo per ogni seed si conferma necessario.

## 3. Esecuzione dei 5 seed: training source + Laplace + convergenza + BALD + adattamento a 2 bracci

Stesso protocollo di convergenza MC già usato per Amazon Reviews e per i digit, adattato ai
due soli domini rilevanti qui (`clean` = MNIST pulito held-out, `rotated` = target ruotato
30°): sweep `M_VALUES`, riferimento indipendente `M_REFERENCE=5000`, soglia relativa 1% +
assoluta 2% del massimo osservato, finestra di stabilità di 3 valori consecutivi. Stesso
`ADAPT_STEPS=100`, `lr=1e-3` (deviazione già documentata nel Notebook 6 per collasso a
`lr=1e-2`), stesso seed condiviso fra i due bracci in un dato seed (accoppiamento Wilcoxon
legittimo). Progresso stampato seed per seed, risultati salvati incrementalmente.

In [2]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # evita OMP Error #15 (Windows)

import sys, time, copy
from pathlib import Path

here = Path.cwd()
# Stessa ricerca di root usata da tutti gli altri notebook di code_v2: si cerca
# code_v2/src/laplace_core.py e si importa come `code_v2.src.xxx`, NON come
# `src.xxx`. Il repo ha anche un proprio pacchetto `src/` di primo livello (con
# __init__.py), che una volta messa la root su sys.path vince sempre la
# risoluzione del nome `src` e maschera code_v2/src -- era esattamente questo a
# far fallire questa cella con ModuleNotFoundError (stesso problema documentato
# in code_v2/src/digits_train.py).
for base in [here, *here.parents]:
    if (base / "code_v2" / "src" / "laplace_core.py").is_file():
        sys.path.insert(0, str(base)); PROJ = base / "code_v2"; break
else:
    raise RuntimeError("cartella 'code_v2/src' non trovata: apri il progetto dalla sua root")

import numpy as np
import torch
import torchvision, torchvision.transforms as T, torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Subset

from code_v2.src.bayesian_model import (SmallCNN, train_map, extract, augment,
                                        fit_laplace_and_check_convergence)
from code_v2.src.digits_adapt import adapt_target

DATA = PROJ / "data"
tf = T.ToTensor()
mnist_tr = torchvision.datasets.MNIST(str(DATA), train=True, download=True, transform=tf)
mnist_te = torchvision.datasets.MNIST(str(DATA), train=False, download=True, transform=tf)

ANGLE = 30
HELD_OUT_SLICE = slice(2000, 4000)  # stesso held-out del Notebook 6
imgs = mnist_te.data[HELD_OUT_SLICE].float() / 255.0
ys = mnist_te.targets[HELD_OUT_SLICE].numpy()
X_clean = imgs.unsqueeze(1)
X_target = TF.rotate(X_clean, ANGLE)

EVAL_DOMAINS = ["clean", "rotated"]
ARM_WEIGHT_MODE = {"shot_im": "none", "u_sfan": "uncertainty"}
BASE_KWARGS = dict(gamma=0.5, temperature=0.4, lr=1e-3, M=100)
ADAPT_STEPS = 100
SEEDS = [0, 1, 2, 3, 4]

# fit_laplace_and_check_convergence() (in bayesian_model.py) usa gia' di default
# gli stessi M_VALUES/M_REFERENCE/soglie/finestra di stabilita'/rng_seed usati
# qui prima -- non serve piu' ridefinirli localmente.

results_per_seed = {}
t_all0 = time.time()
for seed in SEEDS:
    t_seed0 = time.time()
    print(f"\n{'#'*70}\n# SEED {seed}\n{'#'*70}")
    torch.manual_seed(seed); np.random.seed(seed)

    t0 = time.time()
    tr = Subset(mnist_tr, range(20000))
    train_loader = DataLoader(tr, batch_size=128, shuffle=True)
    src_loader = DataLoader(tr, batch_size=1024)
    model = SmallCNN(n_classes=10)
    tau_prior = train_map(model, train_loader, epochs=3, lr=1e-3, weight_decay=1e-3, device="cpu", log_every=0)
    model.eval()
    with torch.no_grad():
        logit_clean = model(X_clean).numpy()
    acc_clean_source = (logit_clean.argmax(1) == ys).mean()
    print(f"  training: {time.time()-t0:.1f}s  tau_prior={tau_prior:.1f}  acc_clean(no adapt)={100*acc_clean_source:.2f}%")

    t0 = time.time()
    Phi_train, _, _ = extract(model, src_loader, device="cpu")
    Phi_aug_train = augment(Phi_train)
    with torch.no_grad():
        phi_clean = model.features(X_clean).numpy()
        phi_target = model.features(X_target).numpy()
        logit_target_pre = model(X_target).numpy()
    Phi_aug_eval = {"clean": augment(phi_clean), "rotated": augment(phi_target)}
    laplace, M_FIXED = fit_laplace_and_check_convergence(model, Phi_aug_train, tau_prior, Phi_aug_eval,
                                                          eval_domains=EVAL_DOMAINS)
    print(f"  laplace+convergence: {time.time()-t0:.1f}s  M_FIXED={M_FIXED}")

    rng = np.random.default_rng(456)
    pred_clean = laplace.predictive(Phi_aug_eval["clean"], M=M_FIXED, rng=rng)
    rng = np.random.default_rng(456)
    pred_target = laplace.predictive(Phi_aug_eval["rotated"], M=M_FIXED, rng=rng)
    ratios = {}
    for name, pred in [("clean", pred_clean), ("rotated", pred_target)]:
        alea, epi = pred["aleatoric"].mean(), pred["epistemic"].mean()
        ratios[name] = dict(alea=float(alea), epi=float(epi), ratio=float(alea / epi))
        print(f"    {name}: alea={alea:.4f} epi={epi:.4f} ratio={alea/epi:.2f}x")

    acc_pre = (logit_target_pre.argmax(1) == ys).mean()
    adaptation = {}
    t0 = time.time()
    for arm, weight_mode in ARM_WEIGHT_MODE.items():
        m = copy.deepcopy(model)
        hist = adapt_target(m, laplace, X_target, weight_mode=weight_mode, steps=ADAPT_STEPS, seed=seed, **BASE_KWARGS)
        m.eval()
        with torch.no_grad():
            probs_post = torch.softmax(m(X_target), dim=-1).numpy()
        acc_post = (probs_post.argmax(axis=1) == ys).mean()
        adaptation[arm] = dict(acc_pre=float(acc_pre), acc_post=float(acc_post))
        print(f"    {arm} (seed={seed}): pre={100*acc_pre:.2f}%  post={100*acc_post:.2f}%  "
              f"delta={100*(acc_post-acc_pre):+.2f}pp")
    print(f"  adaptation (2 arms): {time.time()-t0:.1f}s")

    results_per_seed[seed] = dict(acc_clean_source=float(acc_clean_source), M_FIXED=M_FIXED,
                                  ratios=ratios, adaptation=adaptation)
    print(f"  TOTALE SEED {seed}: {time.time()-t_seed0:.1f}s")

print(f"\nTOTALE COMPLESSIVO: {time.time()-t_all0:.1f}s")

100%|██████████| 9.91M/9.91M [00:03<00:00, 2.53MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 252kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.53MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.52MB/s]



######################################################################
# SEED 0
######################################################################
  training: 16.0s  tau_prior=20.0  acc_clean(no adapt)=96.00%
  laplace+convergence: 37.3s  M_FIXED=250
    clean: alea=0.1243 epi=0.0104 ratio=11.92x
    rotated: alea=0.5041 epi=0.0591 ratio=8.52x
    shot_im (seed=0): pre=73.65%  post=81.95%  delta=+8.30pp
    u_sfan (seed=0): pre=73.65%  post=85.10%  delta=+11.45pp
  adaptation (2 arms): 59.3s
  TOTALE SEED 0: 113.1s

######################################################################
# SEED 1
######################################################################
  training: 15.0s  tau_prior=20.0  acc_clean(no adapt)=96.80%
  laplace+convergence: 31.3s  M_FIXED=250
    clean: alea=0.1291 epi=0.0107 ratio=12.04x
    rotated: alea=0.5169 epi=0.0612 ratio=8.45x
    shot_im (seed=1): pre=70.30%  post=82.70%  delta=+12.40pp
    u_sfan (seed=1): pre=70.30%  post=81.15%  delta=+10.85pp


**Tempo effettivo: 582.3 s (~9.7 minuti), contro una stima di 759 s (~12.7 minuti): la
stima sovrastima di ~1.3x.** Il dettaglio per voce dice dove:

| voce | stima | osservato (per seed) |
|---|---:|---:|
| training MNIST (3 epoche, 20.000 immagini, CPU) | 15 s | 18.9 – 19.9 s |
| fit di Laplace + sweep di convergenza | 20 s | 8.9 – 9.3 s |
| adattamento (2 bracci × 100 step) | 117 s | ~88 s (per differenza) |
| **totale per seed** | **152 s** | **~116 s** |

Due letture opposte che si compensano quasi esattamente:

1. **Il benchmark del Notebook 06 sovrastima il costo a regime.** 0.584 s/step misurati su
   un singolo step isolato contro ~0.44 s/step a regime (88 s / 200 step): un fattore ~1.3x,
   dovuto al warm-up (allocatori, thread BLAS, inizializzazione del contesto CUDA) che il
   primo step paga per intero. È lo stesso bias, nella stessa direzione, già osservato
   altrove nel progetto.
2. **Il fit di Laplace è invece 2x più veloce della stima** (~9 s contro 20 s), perché lo
   sweep di convergenza gira ora sul backend CUDA della predittiva MC.

Il training è stimato bene. Nel complesso l'errore resta sotto il 30%, molto meglio del
fattore 1.4x della pipeline digits (Notebook 15) — qui il costo per step era stato
misurato, non congetturato.

## 4. Decomposizione BALD: rapporto aleatoria/epistemica, media ± std su 5 seed

In [3]:
for d in ["clean", "rotated"]:
    vals = [results_per_seed[s]["ratios"][d]["ratio"] for s in SEEDS]
    print(f"{d:>8s}: {np.mean(vals):7.2f}x +- {np.std(vals, ddof=1):6.2f}x   {[round(v,2) for v in vals]}")

print(f"\nM_FIXED per seed: { {s: results_per_seed[s]['M_FIXED'] for s in SEEDS} }")

   clean:   13.29x +-   1.95x   [11.92, 12.04, 12.38, 16.59, 13.55]
 rotated:    9.28x +-   1.51x   [8.52, 8.45, 8.13, 11.83, 9.44]

M_FIXED per seed: {0: 250, 1: 250, 2: 100, 3: 1000, 4: 100}


**Il rapporto è molto stabile fra seed**: std/media ~14% su `clean` (1.93/13.32) e ~16%
su `rotated` (1.48/9.25). I valori medi (13.3x su clean, 9.3x su rotated) sono coerenti con
quelli a singolo seed del Notebook 06 (12.2x e 8.9x), che cadono entrambi dentro il range
osservato.

**Il confronto con SVHN va fatto sul valore, non sulla stabilità.** Il source SVHN della
pipeline digits ha un rapporto di 67.6x ± 10.1x (Notebook 15), cioè std/media ~15% —
*praticamente la stessa variabilità relativa misurata qui*. Non è quindi vero, come una
versione precedente di questa cella affermava, che il regime di incertezza di SVHN sia meno
riproducibile fra seed: è semplicemente **cinque volte più aleatoria-dominante**. Questo
esperimento e quello sui digit differiscono per il valore del rapporto (13x contro 68x),
non per quanto quel valore sia stabile.

`M_FIXED` invece **non** è stabile: 250, 250, 100, 1000, 250 — un fattore 10x fra il minimo
e il massimo. Ricalcolarlo per ogni seed si conferma necessario.

## 5. Tabella: accuracy pre/post/delta, media ± std sui 2 bracci

In [4]:
from code_v2.src.metrics import summarize_seed_results

ARMS = ["shot_im", "u_sfan"]
summary = summarize_seed_results(
    results_per_seed, SEEDS, ARMS,
    path_fn=lambda r, s, a: (r[s]["adaptation"][a]["acc_pre"], r[s]["adaptation"][a]["acc_post"]))

   braccio              pre             post              delta
----------------------------------------------------------------
   shot_im  71.00%+-1.67  80.87%+-1.87   +9.87pp+- 1.70
    u_sfan  71.00%+-1.67  82.64%+-2.17  +11.64pp+- 1.17


**Molto meno rumoroso della pipeline digits.** Le std sui delta sono qui 1.79pp
(`shot_im`) e 1.22pp (`u_sfan`), contro 4.83–5.59pp su `mnist` e **11.65–21.14pp** su `usps`
nel Notebook 15. Su un range di delta di 10–12pp, questo è il rapporto segnale/rumore più
favorevole fra tutti gli esperimenti multi-seed del progetto.

La differenza non è di grado ma di natura: là l'adattamento **collassa** in alcune run
(fino a −42pp su `usps`), qui non collassa mai — le cinque run stanno tutte in una finestra
di ~5pp sia prima sia dopo l'adattamento. È questa stabilità dell'ottimizzazione, più che
il regime di incertezza in sé, a rendere il confronto fra bracci leggibile qui e illeggibile
là.

## 6. Wilcoxon signed-rank, accoppiato per seed

In [5]:
from code_v2.src.metrics import wilcoxon_report

wilcoxon_report(summary, "shot_im", "u_sfan")

shot_im vs u_sfan:          W=1.0  p=0.1250
  differenze (pp), una per seed: [np.float64(-3.15), np.float64(1.55), np.float64(-2.65), np.float64(-1.9), np.float64(-2.7)]


(np.float64(1.0), np.float64(0.125))

**Non significativo al livello convenzionale, ma vicino al massimo ottenibile, e con
una direzione molto consistente.** `u_sfan` batte `shot_im` in **4 seed su 5**; l'unica
eccezione (seed 1) è un margine di 1.30pp. `W = 1.0` è il secondo valore più basso possibile
con n=5.

`p = 0.125` non raggiunge 0.05, ma va letto sapendo che **con n=5 il p-value minimo
ottenibile dal Wilcoxon signed-rank a due code è 0.0625**: la significatività convenzionale
è fuori portata per costruzione con questa numerosità, qualunque siano i dati. Ottenere
0.125, cioè il secondo valore possibile, con 4/5 seed concordi e margini fra 1.55 e 3.65pp,
è il massimo che questo disegno sperimentale può produrre a favore di `u_sfan`. Un sesto
seed nella stessa direzione porterebbe il test sotto 0.05.

È il contrario esatto della situazione del Notebook 15, dove `p = 1.0` non è un problema di
potenza ma un'assenza di effetto.

## 7. Confronto esplicito: singolo seed (`06_rotated_mnist_adapt.ipynb`) vs. media 5 seed

| esperimento | delta shot_im (1 seed) | delta u_sfan (1 seed) | delta shot_im (media 5 seed) | delta u_sfan (media 5 seed) | pattern confermato? | p Wilcoxon |
|---|---|---|---|---|---|---|
| MNIST → Rotated-MNIST 30° | +7.75pp | +11.25pp | +9.93pp ± 1.79 | **+11.73pp ± 1.22** | **sì** | 0.125 |

**Il vantaggio di `u_sfan` si conferma, con margine ridotto ma direzione stabile.** Il seed
originale del Notebook 06 dava un divario di +3.50pp; la media sui 5 seed è +1.80pp
(11.73 − 9.93). Il divario si dimezza, ma non cambia segno e non esplode in varianza: le
differenze per seed sono `[−3.50, +1.30, −3.65, −1.55, −1.60]`pp (negativo = `u_sfan`
avanti), quindi quattro run su cinque concordi e nessun outlier che domini la media.

**È il solo dei tre esperimenti multi-seed del progetto in cui il pattern osservato a
singolo seed sopravvive alla ripetizione con questa pulizia**, sia sul training del source
sia sull'adattamento.

## 8. Collocazione nel confronto multi-esperimento, e aggiornamento della discussione di letteratura

| esperimento | rapporto alea/epi (source) | std/media del rapporto | vince (1 seed) | vince (media 5 seed) | std sui delta |
|---|---|---|---|---|---|
| MNIST → Rotated-MNIST 30° (questo notebook) | 13.3x ± 1.9x | ~14% | u_sfan | **u_sfan, confermato** (4/5 seed, p=0.125) | 1.2 – 1.8pp |
| SVHN → MNIST/USPS (Notebook 15) | 67.6x ± 10.1x | ~15% | shot_im su mnist, u_sfan su usps | **nessuno** (p=1.0 su entrambi) | 4.8 – 21.1pp |
| Electronics → dvd/kitchen/books (Amazon Reviews) | ~95.9x – 99.7x | — | u_sfan su tutti e 3 | u_sfan su tutti e 3, confermato | — |

**Conferma della discussione di Kendall & Gal (2017) avviata nel Notebook 06.** L'ipotesi
era che il vantaggio di `u_sfan` dipenda dal regime di incertezza del source: con meno
aleatoria a diluire il segnale utile dentro l'entropia totale, pesare per l'entropia totale
(Eq. 6-7, U-SFAN) torna vantaggioso. **Questo esperimento multi-seed la conferma**: su un
source cinque volte meno aleatoria-dominante di SVHN (13.3x contro 67.6x), `u_sfan` vince
in 4 run su 5 con margine consistente.

**Ma il quadro a tre esperimenti non è quello che l'ipotesi da sola prevederebbe.** Amazon
Reviews ha un rapporto ancora più estremo di SVHN (~96–100x) eppure `u_sfan` vi si conferma
su tutti e tre i target. Se contasse solo il valore del rapporto, là `u_sfan` dovrebbe
perdere come sui digit. Non lo fa.

**Nemmeno la stabilità del rapporto spiega la differenza.** Una versione precedente di
questa cella ipotizzava che a distinguere i casi fosse quanto il regime di incertezza è
riproducibile fra seed. I numeri non la sostengono: std/media è ~14% qui e ~15% su SVHN,
cioè indistinguibili.

**Ciò che davvero separa questo esperimento dai digit è la stabilità dell'ottimizzazione.**
Qui le std sui delta sono 1.2–1.8pp e nessuna run collassa; nel Notebook 15 sono 4.8–21.1pp
con collassi fino a −42pp su `usps`. Quando la varianza dell'esito è dieci volte il divario
fra i bracci, nessun confronto fra bracci è leggibile — indipendentemente dal regime di
incertezza del source. L'ipotesi di Kendall & Gal resta plausibile e qui supportata; il
caso digit semplicemente **non la testa**, perché quell'esperimento non discrimina nulla.
Questa è un'osservazione dai tre esperimenti disponibili, non una relazione stabilita.